# WP16 prospective N13 frozen K36 holdout
The code and all three pass criteria were frozen before N13. Run this single cell once in Colab; retain failures without retuning.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import os
import subprocess
import sys
import tempfile

ROOT = "/content/drive/MyDrive/WP16_CUTOFF_ESCALATION"
N12 = ROOT + "/wp16_phase_cutoff_escalation_N12.json"
SOURCE = ROOT + "/wp16_036_phase_velocity_rhs_sources.json"
N13 = ROOT + "/wp16_phase_cutoff_escalation_N13.json"
OUT = ROOT + "/wp16_036_N13_frozen_K36_holdout.json"
PIN = "2b36e519f6c6cd22f8d36eca143627e014417220"

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert os.path.isfile(N12), N12
assert os.path.isfile(SOURCE), SOURCE
assert digest(N12) == "ec07d1a263eb43c1a1d6228164ba80a4e29b90b6206bb606c612192a4ee38855"
assert digest(SOURCE) == "193cbb7f485ab54ceed0d5cb38f97f0fc98c197e5f88e288fdbd277627608c8e"
assert not os.path.exists(N13), f"N13 already exists; preserve the first run: {N13}"
assert not os.path.exists(OUT), f"Holdout already exists; preserve the first result: {OUT}"

repo = tempfile.mkdtemp(prefix="navier-stokes-n13-")
subprocess.run(["git", "clone", "https://github.com/reggaesharkk/navier-stokes-bridge-audit.git", repo], check=True)
subprocess.run(["git", "checkout", PIN], cwd=repo, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
assert head == PIN, head
print("Frozen code HEAD:", head, flush=True)

resume = [
    sys.executable, "-u", "src/wp16_phase_cutoff_resume.py",
    "--resume-json", N12, "--output", N13, "--cutoffs", "13",
    "--search-grid", "40", "--new-global-draws", "16",
    "--new-block-rounds", "3", "--full-block-rounds", "4",
    "--block-trials", "72", "--block-size", "40", "--initial-step", "0.30",
]
print("GENERATING N13:", " ".join(resume), flush=True)
subprocess.run(resume, cwd=repo, check=True)

holdout = [
    sys.executable, "-u", "src/wp16_036_N13_frozen_K36_holdout.py",
    "--n12-json", N12, "--n13-json", N13,
    "--source-json", SOURCE, "--output", OUT,
]
print("RUNNING FROZEN N13 K36 HOLDOUT:", " ".join(holdout), flush=True)
subprocess.run(holdout, cwd=repo, check=True)
print("Saved N13:", N13)
print("Saved holdout:", OUT)
